# 09 - Export Models untuk Deploy (Baseline + Robust Top-10)

**Tujuan:** Menghasilkan 2 file model XGBoost Top-10 siap deploy ke AWS untuk testing real-traffic.

**Output:**
- `../models/baseline_xgboost_top10.json` — model baseline (tanpa adversarial training)
- `../models/robust_xgboost_top10.json` — model robust (sudah ada dari notebook 06)
- `../models/deploy_meta.json` — metadata (scaler, label mapping, feature names)

**Catatan:** File ini harus diupload ke S3 agar EC2 Analyzer bisa download.

In [ ]:
import sys
!{sys.executable} -m pip install xgboost scikit-learn numpy pandas -q

In [ ]:
import os
import json
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score
from xgboost import XGBClassifier

DATA_DIR = '../data/'
MODEL_DIR = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_SEED = 42
TEST_SIZE = 0.20

print('Libraries loaded ✓')

## 1. Load Data & Feature Info

In [ ]:
# Load experiment results (contains feature names, label mapping)
with open(os.path.join(DATA_DIR, 'experiment_results_03.pkl'), 'rb') as f:
    exp = pickle.load(f)

# Load cleaned dataset
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'rb') as f:
    data = pickle.load(f)

# Extract info
top10_features = exp['top10_features']
all_features = exp['feature_names']
label_mapping = exp['label_mapping']

print(f'Dataset shape: {data["X"].shape}')
print(f'Top-10 features: {top10_features}')
print(f'Label mapping: {label_mapping}')
print(f'Scaler available: {"scaler" in data}')

## 2. Prepare Top-10 Data

In [ ]:
# Select Top-10 features
idx_top10 = [all_features.index(f) for f in top10_features]
X_top10 = data['X'][:, idx_top10]
y = data['y']

# Split (same split as all other notebooks)
X_train, X_test, y_train, y_test = train_test_split(
    X_top10, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

print(f'X_top10 shape: {X_top10.shape}')
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

## 3. Train & Export Baseline Model (Top-10)

In [ ]:
import time

# Train baseline (same hyperparameters as robust for fair comparison)
n_classes = len(np.unique(y))

model_baseline = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=n_classes,
    eval_metric='mlogloss',
    random_state=RANDOM_SEED,
    n_jobs=-1,
    tree_method='hist'
)

print('Training Baseline XGBoost Top-10...')
start = time.time()
model_baseline.fit(X_train, y_train)
train_time = time.time() - start
print(f'Done in {train_time:.1f}s')

# Evaluate
y_pred = model_baseline.predict(X_test)
mcc = matthews_corrcoef(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f'Baseline Performance: MCC={mcc:.4f} | F1={f1*100:.2f}%')

# Save
baseline_path = os.path.join(MODEL_DIR, 'baseline_xgboost_top10.json')
model_baseline.save_model(baseline_path)
baseline_size = os.path.getsize(baseline_path) / (1024*1024)
print(f'\nSaved: {baseline_path} ({baseline_size:.2f} MB)')

## 4. Verify Robust Model Exists

In [ ]:
# Check robust model
robust_path = os.path.join(MODEL_DIR, 'robust_xgboost_top10.json')

if os.path.exists(robust_path):
    robust_size = os.path.getsize(robust_path) / (1024*1024)
    print(f'✓ Robust model exists: {robust_path} ({robust_size:.2f} MB)')
    
    # Quick verify
    model_robust = XGBClassifier()
    model_robust.load_model(robust_path)
    y_pred_r = model_robust.predict(X_test)
    mcc_r = matthews_corrcoef(y_test, y_pred_r)
    f1_r = f1_score(y_test, y_pred_r, average='weighted', zero_division=0)
    print(f'  Robust Performance (clean data): MCC={mcc_r:.4f} | F1={f1_r*100:.2f}%')
else:
    print(f'✗ Robust model NOT FOUND at {robust_path}')
    print('  → Run notebook 06_adversarial_training.ipynb first!')

## 5. Export Metadata (Scaler + Label Mapping + Feature Names)

In [ ]:
# Extract scaler values for Top-10 features only
scaler = data['scaler']
scaler_mean_top10 = scaler.mean_[idx_top10].tolist()
scaler_scale_top10 = scaler.scale_[idx_top10].tolist()

# Build deploy metadata
deploy_meta = {
    'feature_names': top10_features,
    'n_features': len(top10_features),
    'n_classes': n_classes,
    'label_mapping': label_mapping,
    'inverse_label_mapping': {str(v): k for k, v in label_mapping.items()},
    'scaler': {
        'type': 'StandardScaler',
        'mean': scaler_mean_top10,
        'scale': scaler_scale_top10
    },
    'models': {
        'baseline': 'baseline_xgboost_top10.json',
        'robust': 'robust_xgboost_top10.json'
    },
    'dataset': 'CSE-CIC-IDS2018',
    'description': 'XGBoost Top-10 features for NIDS01 real-traffic testing'
}

meta_path = os.path.join(MODEL_DIR, 'deploy_meta.json')
with open(meta_path, 'w') as f:
    json.dump(deploy_meta, f, indent=2)

print(f'Saved: {meta_path}')
print(f'\nFeatures: {top10_features}')
print(f'Classes: {label_mapping}')

## 6. Summary & Upload Instructions

In [ ]:
print('='*60)
print('  MODEL EXPORT COMPLETE')
print('='*60)
print(f'''
Files ready for deployment:

  1. {baseline_path}
     → Baseline XGBoost Top-10 ({baseline_size:.2f} MB)
     → MCC={mcc:.4f} on clean test data

  2. {robust_path}
     → Robust XGBoost Top-10 ({robust_size:.2f} MB)
     → MCC={mcc_r:.4f} on clean test data

  3. {meta_path}
     → Scaler, label mapping, feature names

Upload to S3:
  aws s3 cp ../models/baseline_xgboost_top10.json s3://ssh-detection-features-232032302717/models/nids01/
  aws s3 cp ../models/robust_xgboost_top10.json s3://ssh-detection-features-232032302717/models/nids01/
  aws s3 cp ../models/deploy_meta.json s3://ssh-detection-features-232032302717/models/nids01/

Verify:
  aws s3 ls s3://ssh-detection-features-232032302717/models/nids01/
''')